# Notebook-first application walkthrough

**Problem / objective:** Estimate residential transaction values with time-aware validation rather than a random split that leaks future market conditions.

**Decision / solution:** Use the model as a valuation-screening tool, expose uncertainty and error slices, and avoid pretending the estimate is a formal survey or valuation.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'uk_house_price_prediction'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Use the model as a valuation-screening tool, expose uncertainty and error slices, and avoid pretending the estimate is a formal survey or valuation.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# 01 — UK House Price Analysis and Prediction

**Goal:** analyse 2025–2026 registered property sales in England and Wales, build a forward-looking price model, quantify uncertainty and document where the model should not be trusted.

[Open in Google Colab](https://colab.research.google.com/github/Jorgoluka100/uni_projects/blob/main/01_UK_House_Price_Analysis_and_Prediction.ipynb)

This is the first notebook in a rebuilt, current-data portfolio. It combines Python, Pandas, NumPy, DuckDB SQL, Matplotlib, Seaborn, leakage-safe preprocessing, CatBoost and automated acceptance tests. Every metric is generated by this execution.

## Decision framing

A property analyst needs a reproducible market view and an initial valuation range for ordinary residential transactions. The model is evaluated as a genuine future test:

- **Train:** January–September 2025
- **Validation:** October–December 2025
- **Untouched test:** January–June 2026

This is not a surveyor valuation, mortgage decision or investment recommendation. The data omit bedrooms, floor area, condition and exact property characteristics, so predictions are broad market estimates.

In [1]:
# Colab setup
import importlib.util, subprocess, sys
missing=[p for p in ['duckdb','catboost'] if importlib.util.find_spec(p) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q','duckdb==1.3.2','catboost==1.2.8'])

import json, os, random, urllib.request
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED=42; random.seed(SEED); np.random.seed(SEED); sns.set_theme(style='whitegrid')
ROOT=Path('/content/house_price_project' if Path('/content').exists() else './house_price_project'); ROOT.mkdir(parents=True,exist_ok=True)
print({'python':sys.version.split()[0],'duckdb':duckdb.__version__,'seed':SEED})

{'python': '3.12.13', 'duckdb': '1.3.2', 'seed': 42}


## 1. Official 2025–2026 data and licence

Source: [HM Land Registry Price Paid Data](https://www.gov.uk/government/statistical-data-sets/price-paid-data-downloads), updated monthly. It covers property sales in England and Wales sold for value and lodged for registration.

The 2025 yearly file is combined with the available January–June 2026 file. HM Land Registry states that current data are revised as registrations arrive; therefore 2026 volumes are explicitly labelled **provisional** and are not treated as complete market activity.

Attribution: *Contains HM Land Registry data © Crown copyright and database right 2021. This data is licensed under the Open Government Licence v3.0.* Address fields have additional permitted-use conditions; this educational notebook uses coarse geography and does not expose exact addresses.

In [2]:
URLS={2025:'https://price-paid-data.publicdata.landregistry.gov.uk/pp-2025.csv',2026:'https://price-paid-data.publicdata.landregistry.gov.uk/pp-2026.csv'}
paths={}
for year,url in URLS.items():
    path=ROOT/f'pp-{year}.csv'; paths[year]=path
    if not path.exists(): urllib.request.urlretrieve(url,path)
print({year:round(path.stat().st_size/1e6,1) for year,path in paths.items()})

con=duckdb.connect(database=':memory:')
file_list=','.join(repr(str(p)) for p in paths.values())
con.execute(f'''CREATE VIEW price_paid AS SELECT
 column00 AS transaction_id, CAST(column01 AS BIGINT) AS price,
 CAST(strptime(column02,'%Y-%m-%d %H:%M') AS DATE) AS transfer_date,
 NULLIF(trim(column03),'') AS postcode, column04 AS property_type,
 column05 AS old_new, column06 AS duration, column11 AS town_city,
 column12 AS district, column13 AS county, column14 AS ppd_category,
 column15 AS record_status
 FROM read_csv([{file_list}],header=false,all_varchar=true)''')
source_audit=con.execute('''SELECT COUNT(*) raw_rows,COUNT(DISTINCT transaction_id) unique_ids,
 MIN(transfer_date) min_date,MAX(transfer_date) max_date,
 SUM(postcode IS NULL) missing_postcodes,MIN(price) min_price,MAX(price) max_price
 FROM price_paid''').df()
display(source_audit.T)
assert int(source_audit.raw_rows.iloc[0])==1_184_740
assert int(source_audit.raw_rows.iloc[0])==int(source_audit.unique_ids.iloc[0])

{2025: 162.4, 2026: 43.8}
                                     0
raw_rows                       1184740
unique_ids                     1184740
min_date           2025-01-01 00:00:00
max_date           2026-06-30 00:00:00
missing_postcodes               2518.0
min_price                            1
max_price                    793020000


## 2. Data contract and modelling population

The valuation population is deliberately narrower than the raw registry:

- ordinary residential types: detached, semi-detached, terraced and flats;
- Category A standard transactions;
- prices from £20,000 to £5,000,000;
- complete postcode;
- transfer dates from January 2025 through June 2026.

Other property, unusual transfers and extreme values remain part of the market audit but are not forced into a residential valuation model.

In [3]:
quality=con.execute('''SELECT
 COUNT(*) raw,
 SUM(property_type IN ('D','S','T','F')) ordinary_residential,
 SUM(ppd_category='A') standard_category,
 SUM(price BETWEEN 20000 AND 5000000) plausible_price,
 SUM(postcode IS NOT NULL) postcode_present,
 SUM(property_type IN ('D','S','T','F') AND ppd_category='A' AND price BETWEEN 20000 AND 5000000 AND postcode IS NOT NULL) accepted
 FROM price_paid''').df()
display(quality.T)
accepted_rows=int(quality.accepted.iloc[0]); raw_rows=int(quality.raw.iloc[0]); print({'accepted_rows':accepted_rows,'accepted_pct':accepted_rows/raw_rows})

con.execute('''CREATE VIEW residential AS SELECT *,
 split_part(postcode,' ',1) AS postcode_district,
 regexp_extract(postcode,'^[A-Z]+',0) AS postcode_area,
 EXTRACT(year FROM transfer_date)::INTEGER AS transfer_year,
 EXTRACT(month FROM transfer_date)::INTEGER AS transfer_month,
 sin(2*pi()*EXTRACT(month FROM transfer_date)/12) AS month_sin,
 cos(2*pi()*EXTRACT(month FROM transfer_date)/12) AS month_cos
 FROM price_paid WHERE property_type IN ('D','S','T','F') AND ppd_category='A'
 AND price BETWEEN 20000 AND 5000000 AND postcode IS NOT NULL''')
split_counts=con.execute('''SELECT CASE WHEN transfer_date<'2025-10-01' THEN 'train'
 WHEN transfer_date<'2026-01-01' THEN 'validation' ELSE 'test_2026' END split,COUNT(*) row_count,
 median(price) median_price FROM residential GROUP BY 1 ORDER BY 1''').df()
display(split_counts); assert accepted_rows==995_059

                              0
raw                   1184740.0
ordinary_residential  1132583.0
standard_category      995852.0
plausible_price       1178936.0
postcode_present      1182222.0
accepted               995059.0
{'accepted_rows': 995059, 'accepted_pct': 0.8398965173793406}
        split  row_count  median_price
0   test_2026     216564      285000.0
1       train     595617      298000.0
2  validation     182878      292000.0


## 3. SQL market analysis

In [4]:
monthly=con.execute('''SELECT date_trunc('month',transfer_date) sale_month,COUNT(*) transactions,
 median(price) median_price,avg(price) mean_price,
 quantile_cont(price,0.25) p25,quantile_cont(price,0.75) p75
 FROM residential GROUP BY 1 ORDER BY 1''').df()
property_summary=con.execute('''SELECT property_type,COUNT(*) transactions,median(price) median_price,
 quantile_cont(price,0.25) p25,quantile_cont(price,0.75) p75
 FROM residential GROUP BY 1 ORDER BY median_price DESC''').df()
county_summary=con.execute('''SELECT county,COUNT(*) transactions,median(price) median_price
 FROM residential WHERE transfer_date<'2026-01-01' GROUP BY 1 HAVING COUNT(*)>=1000
 ORDER BY median_price DESC LIMIT 15''').df()
display(property_summary); display(county_summary)
fig,axs=plt.subplots(1,2,figsize=(14,5)); sns.lineplot(data=monthly,x='sale_month',y='median_price',marker='o',ax=axs[0]); axs[0].axvline(pd.Timestamp('2026-01-01'),ls='--',color='black',label='Provisional 2026'); axs[0].legend(); axs[0].set(title='Median registered sale price',ylabel='Price (£)'); sns.barplot(data=property_summary,x='property_type',y='median_price',ax=axs[1]); axs[1].set(title='Median price by property type',ylabel='Price (£)'); plt.tight_layout(); plt.show()

  property_type  transactions  median_price       p25       p75
0             D        255168      420000.0  318000.0  587500.0
1             S        300494      275000.0  205000.0  380000.0
2             T        277837      240000.0  165000.0  352500.0
3             F        161560      235000.0  150000.0  380000.0
                          county  transactions  median_price
0         WINDSOR AND MAIDENHEAD          2047      545000.0
1                 GREATER LONDON         82801      528000.0
2                         SURREY         16916      510000.0
3                      WOKINGHAM          2544      490000.0
4                BUCKINGHAMSHIRE          7768      460790.0
5                  HERTFORDSHIRE         15676      460000.0
6               BRACKNELL FOREST          1782      424000.0
7              BRIGHTON AND HOVE          3726      415000.0
8                    OXFORDSHIRE          9974      410000.0
9   BATH AND NORTH EAST SOMERSET          2739      400000.0
10       

## 4. Leakage-safe feature table

Exact address fields and transaction identifiers are excluded. Coarse location and transaction attributes are retained. All fallback medians and uncertainty thresholds are learned from training or validation data only.

In [5]:
model_df=con.execute('''SELECT price,transfer_date,postcode_district,postcode_area,
 property_type,old_new,duration,town_city,district,county,transfer_year,transfer_month,month_sin,month_cos
 FROM residential ORDER BY transfer_date,transaction_id''').df()
categorical=['postcode_district','postcode_area','property_type','old_new','duration','town_city','district','county']
numeric=['transfer_year','transfer_month','month_sin','month_cos']; features=categorical+numeric
for c in categorical: model_df[c]=model_df[c].fillna('UNKNOWN').astype(str)
train=model_df[model_df.transfer_date<pd.Timestamp('2025-10-01')].copy()
validation=model_df[(model_df.transfer_date>=pd.Timestamp('2025-10-01'))&(model_df.transfer_date<pd.Timestamp('2026-01-01'))].copy()
test=model_df[model_df.transfer_date>=pd.Timestamp('2026-01-01')].copy()
assert train.transfer_date.max()<validation.transfer_date.min()<test.transfer_date.min()
print({'train':len(train),'validation':len(validation),'test':len(test),'features':features})

{'train': 595617, 'validation': 182878, 'test': 216564, 'features': ['postcode_district', 'postcode_area', 'property_type', 'old_new', 'duration', 'town_city', 'district', 'county', 'transfer_year', 'transfer_month', 'month_sin', 'month_cos']}


## 5. Honest baselines

In [6]:
global_median=float(train.price.median())
type_median=train.groupby('property_type').price.median()
area_type_median=train.groupby(['postcode_district','property_type']).price.median()
def baseline_predict(frame):
    keys=pd.MultiIndex.from_frame(frame[['postcode_district','property_type']])
    area=np.asarray(area_type_median.reindex(keys),dtype=float)
    fallback=frame.property_type.map(type_median).fillna(global_median).to_numpy(float)
    return np.where(np.isfinite(area),area,fallback)
baseline_val=baseline_predict(validation); baseline_test=baseline_predict(test)

def regression_metrics(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    return {'MAE':mean_absolute_error(y,p),'RMSE':mean_squared_error(y,p)**.5,'R2':r2_score(y,p),'MAPE':np.mean(np.abs((y-p)/y))*100,'within_20pct':np.mean(np.abs(y-p)/y<=.20)}
display(pd.DataFrame({'Global median':regression_metrics(test.price,np.repeat(global_median,len(test))),'Area + property baseline':regression_metrics(test.price,baseline_test)}).T.round(3))

                                 MAE        RMSE     R2    MAPE  within_20pct
Global median             152553.263  249557.995 -0.035  54.356         0.282
Area + property baseline   82804.352  154479.148  0.604  25.341         0.560


## 6. CatBoost price model

CatBoost handles categorical features without one-hot exploding thousands of postcode districts. The target is `log1p(price)`, reducing the influence of the long luxury-property tail. Early stopping is based only on late-2025 validation data.

In [7]:
model=CatBoostRegressor(iterations=500,depth=9,learning_rate=.08,loss_function='MAE',eval_metric='MAE',l2_leaf_reg=8,random_seed=SEED,allow_writing_files=False,thread_count=-1,verbose=50)
model.fit(train[features],np.log1p(train.price),cat_features=categorical,eval_set=(validation[features],np.log1p(validation.price)),early_stopping_rounds=50)
pred_val=np.expm1(model.predict(validation[features])).clip(20000,5000000)
pred_test=np.expm1(model.predict(test[features])).clip(20000,5000000)
print({'best_iteration':model.get_best_iteration(),'validation':regression_metrics(validation.price,pred_val)})

0:	learn: 0.4568447	test: 0.4553738	best: 0.4553738 (0)	total: 235ms	remaining: 1m 57s
50:	learn: 0.2578445	test: 0.2580294	best: 0.2580294 (50)	total: 7.54s	remaining: 1m 6s
100:	learn: 0.2507114	test: 0.2518169	best: 0.2518169 (100)	total: 16.2s	remaining: 1m 4s
150:	learn: 0.2468322	test: 0.2487551	best: 0.2487551 (150)	total: 23.9s	remaining: 55.3s
200:	learn: 0.2441387	test: 0.2468646	best: 0.2468646 (200)	total: 32.3s	remaining: 48.1s
250:	learn: 0.2419834	test: 0.2453969	best: 0.2453969 (250)	total: 40.4s	remaining: 40.1s
300:	learn: 0.2404683	test: 0.2445139	best: 0.2445139 (300)	total: 48.2s	remaining: 31.9s
350:	learn: 0.2392658	test: 0.2439101	best: 0.2439101 (350)	total: 55.7s	remaining: 23.7s
400:	learn: 0.2382280	test: 0.2433964	best: 0.2433964 (400)	total: 1m 3s	remaining: 15.7s
450:	learn: 0.2373834	test: 0.2430264	best: 0.2430264 (450)	total: 1m 11s	remaining: 7.76s
499:	learn: 0.2367052	test: 0.2427901	best: 0.2427901 (499)	total: 1m 19s	remaining: 0us

bestTest = 0.2

## 7. Untouched 2026 test evaluation

In [8]:
evaluation=pd.DataFrame({
 'Global median':regression_metrics(test.price,np.repeat(global_median,len(test))),
 'Area + property baseline':regression_metrics(test.price,baseline_test),
 'CatBoost':regression_metrics(test.price,pred_test)}).T
baseline_mae=evaluation.loc['Area + property baseline','MAE']; model_mae=evaluation.loc['CatBoost','MAE']; improvement=1-model_mae/baseline_mae
display(evaluation.round(3)); print({'mae_improvement_vs_strong_baseline_pct':round(improvement*100,2)})

comparison=pd.DataFrame({'actual':test.price.to_numpy(),'prediction':pred_test,'property_type':test.property_type.to_numpy(),'old_new':test.old_new.to_numpy(),'county':test.county.to_numpy()})
comparison['absolute_error']=np.abs(comparison.actual-comparison.prediction); comparison['absolute_pct_error']=comparison.absolute_error/comparison.actual
fig,axs=plt.subplots(1,2,figsize=(13,5)); sample=comparison.sample(min(12000,len(comparison)),random_state=SEED); axs[0].scatter(sample.actual,sample.prediction,s=4,alpha=.2); lim=1_500_000; axs[0].plot([0,lim],[0,lim],'--',color='black'); axs[0].set(xlim=(0,lim),ylim=(0,lim),xlabel='Actual (£)',ylabel='Predicted (£)',title='2026 predictions'); sns.boxplot(data=comparison,x='property_type',y='absolute_pct_error',showfliers=False,ax=axs[1]); axs[1].set(title='Percentage error by property type',ylabel='Absolute percentage error'); plt.tight_layout(); plt.show()

                                 MAE        RMSE     R2    MAPE  within_20pct
Global median             152553.263  249557.995 -0.035  54.356         0.282
Area + property baseline   82804.352  154479.148  0.604  25.341         0.560
CatBoost                   81804.952  154441.649  0.604  24.343         0.561
{'mae_improvement_vs_strong_baseline_pct': np.float64(1.21)}


## 8. Validation-calibrated uncertainty and failure analysis

In [9]:
q90=float(np.quantile(np.abs(validation.price.to_numpy()-pred_val),.90))
lower=np.maximum(20000,pred_test-q90); upper=pred_test+q90
coverage=float(np.mean((test.price.to_numpy()>=lower)&(test.price.to_numpy()<=upper))); avg_width=float(np.mean(upper-lower))
interval_metrics={'nominal_coverage':.90,'test_coverage':coverage,'average_width_pounds':avg_width,'validation_q90_absolute_error':q90}
print(json.dumps(interval_metrics,indent=2))

slice_property=comparison.groupby('property_type').agg(rows=('actual','size'),MAE=('absolute_error','mean'),MAPE=('absolute_pct_error','mean')).sort_values('MAE',ascending=False)
slice_status=comparison.groupby('old_new').agg(rows=('actual','size'),MAE=('absolute_error','mean'),MAPE=('absolute_pct_error','mean')).sort_values('MAE',ascending=False)
slice_county=comparison.groupby('county').agg(rows=('actual','size'),MAE=('absolute_error','mean'),MAPE=('absolute_pct_error','mean')).query('rows>=500').sort_values('MAE',ascending=False).head(12)
display(slice_property.round(3)); display(slice_status.round(3)); display(slice_county.round(3))
display(comparison.nlargest(10,'absolute_error')[['actual','prediction','absolute_error','property_type','county']])

{
  "nominal_coverage": 0.9,
  "test_coverage": 0.9162464675569346,
  "average_width_pounds": 381679.2346557995,
  "validation_q90_absolute_error": 199323.94805716627
}
                rows         MAE   MAPE
property_type                          
D              52783  127609.573  0.260
F              34972   78346.749  0.312
S              65491   65228.905  0.210
T              63318   62676.377  0.226
           rows        MAE   MAPE
old_new                          
Y           251  88996.967  0.190
N        216313  81796.607  0.243
                               rows         MAE   MAPE
county                                                
GREATER LONDON                20836  154678.473  0.275
WINDSOR AND MAIDENHEAD          509  150711.147  0.238
SURREY                         4758  133029.676  0.237
BRIGHTON AND HOVE              1019  130772.353  0.282
BATH AND NORTH EAST SOMERSET    748  128061.477  0.272
BUCKINGHAMSHIRE                2028  122166.610  0.212
OXFORDSHIRE    

## 9. Explainability and verified model reload

In [10]:
importance=pd.DataFrame({'feature':features,'importance':model.get_feature_importance()}).sort_values('importance',ascending=False)
display(importance); fig,ax=plt.subplots(figsize=(8,5)); sns.barplot(data=importance,y='feature',x='importance',ax=ax); ax.set_title('CatBoost feature importance (predictive, not causal)'); plt.show()

model_path=ROOT/'uk_house_price_catboost.cbm'; model.save_model(model_path)
reloaded=CatBoostRegressor(); reloaded.load_model(model_path)
sample_x=test[features].iloc[:500]; original=model.predict(sample_x); restored=reloaded.predict(sample_x)
reload_delta=float(np.max(np.abs(original-restored))); print({'model_path':str(model_path),'size_mb':model_path.stat().st_size/1e6,'max_log_prediction_delta':reload_delta}); assert reload_delta<1e-12

              feature  importance
2       property_type   25.297056
6            district   21.910221
0   postcode_district   16.006521
1       postcode_area   14.138666
7              county    7.163795
4            duration    5.508067
5           town_city    5.117049
3             old_new    3.306294
10          month_sin    0.857580
9      transfer_month    0.452817
11          month_cos    0.241935
8       transfer_year    0.000000
{'model_path': 'house_price_project/uk_house_price_catboost.cbm', 'size_mb': 10.108236, 'max_log_prediction_delta': 0.0}


## 10. Acceptance tests, governance and CV evidence

In [11]:
checks={
 'official_source_rows':raw_rows==1_184_740,
 'transaction_ids_unique':int(source_audit.unique_ids.iloc[0])==raw_rows,
 'data_reaches_june_2026':pd.Timestamp(source_audit.max_date.iloc[0]).date()==pd.Timestamp('2026-06-30').date(),
 'accepted_population_verified':accepted_rows==995_059,
 'chronological_split':train.transfer_date.max()<validation.transfer_date.min()<test.transfer_date.min(),
 'test_is_2026_only':test.transfer_date.min()>=pd.Timestamp('2026-01-01'),
 'predictions_finite':np.isfinite(pred_test).all(),
 'model_beats_area_property_baseline':model_mae<baseline_mae,
 'interval_order_valid':np.all(lower<=pred_test) and np.all(pred_test<=upper),
 'model_reload_exact':reload_delta<1e-12}
display(pd.Series(checks,name='passed').to_frame()); assert all(checks.values())

summary={'source_snapshot':'HM Land Registry 2025 and January-June 2026 files','raw_transactions':raw_rows,'model_population':accepted_rows,'train_rows':len(train),'validation_rows':len(validation),'untouched_2026_test_rows':len(test),'test_metrics':{name:{k:float(v) for k,v in row.items()} for name,row in evaluation.iterrows()},'mae_improvement_vs_strong_baseline_pct':float(improvement*100),**interval_metrics,'model_reload_delta':reload_delta}
print('RUN-DERIVED SUMMARY'); print(json.dumps(summary,indent=2))
print(f"CV bullet: Built a leakage-safe England and Wales property valuation pipeline over {raw_rows:,} official 2025–2026 transactions using DuckDB SQL, Pandas and CatBoost; achieved £{model_mae:,.0f} MAE on {len(test):,} untouched 2026 sales ({improvement:.1%} better than a postcode-district/property baseline), with calibrated intervals and exact model-reload verification.")

model_card={'system':'Residential sale-price analysis and broad valuation support','data':'HM Land Registry Price Paid Data, 2025 and January-June 2026 snapshot','intended_use':'portfolio demonstration and human-reviewed market analysis','not_for':'surveying, mortgage underwriting, automated investment or individual financial decisions','metrics':summary,'limitations':['2026 registrations are provisional and revised after this snapshot','no floor area, bedrooms, condition, energy rating or exact property features','Price Paid Data excludes sales not lodged or not sold for value','uncertainty intervals can under-cover during market or geographic shift','feature importance is predictive association, not causal impact']}
print(json.dumps(model_card,indent=2))

                                    passed
official_source_rows                  True
transaction_ids_unique                True
data_reaches_june_2026                True
accepted_population_verified          True
chronological_split                   True
test_is_2026_only                     True
predictions_finite                    True
model_beats_area_property_baseline    True
interval_order_valid                  True
model_reload_exact                    True
RUN-DERIVED SUMMARY
{
  "source_snapshot": "HM Land Registry 2025 and January-June 2026 files",
  "raw_transactions": 1184740,
  "model_population": 995059,
  "train_rows": 595617,
  "validation_rows": 182878,
  "untouched_2026_test_rows": 216564,
  "test_metrics": {
    "Global median": {
      "MAE": 152553.2633494025,
      "RMSE": 249557.9949089,
      "R2": -0.03467939189598157,
      "MAPE": 54.355630753523144,
      "within_20pct": 0.28174119428898614
    },
    "Area + property baseline": {
      "MAE": 82804.3522

## Conclusion

The final evidence is forward-looking rather than a random split: 2026 sales are never used for training, median fallbacks use training data only, and the uncertainty width is chosen on validation residuals. Production work would join EPC floor area and property attributes under compatible licensing, use rolling monthly backtests, monitor geographic drift and require professional review for any valuation decision.

# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `run.py`


In [ ]:
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd

from src.evaluation import AreaPropertyBaseline, conformal_radius, interval_coverage, regression_metrics

ROOT = Path(__file__).resolve().parent
EVIDENCE = ROOT / "results" / "verified_metrics.json"


def self_test() -> None:
    train = pd.DataFrame(
        {
            "postcode_district": ["E1", "E1", "SW1", "SW1"],
            "property_type": ["F", "F", "T", "T"],
            "price": [300_000, 320_000, 700_000, 760_000],
        }
    )
    test = pd.DataFrame(
        {
            "postcode_district": ["E1", "SW1", "N1"],
            "property_type": ["F", "T", "F"],
            "price": [310_000, 730_000, 330_000],
        }
    )
    baseline = AreaPropertyBaseline().fit(train)
    pred = baseline.predict(test)
    assert np.allclose(pred, [310_000, 730_000, 310_000])
    metrics = regression_metrics(test.price, pred)
    assert metrics["mae"] >= 0
    radius = conformal_radius([100, 200, 300], [90, 190, 250], coverage=0.90)
    assert radius == 50
    covered = interval_coverage([100, 200], [100, 220], radius=30, floor=0, cap=1000)
    assert covered["coverage"] == 1.0
    print("UK house price self-test passed.")


def check_evidence() -> None:
    report = json.loads(EVIDENCE.read_text(encoding="utf-8"))
    assert report["verification_pass"] is True
    assert report["model_population"] == 995_059
    assert report["untouched_2026_test_rows"] == 216_564
    model = report["test_metrics"]["catboost"]
    baseline = report["test_metrics"]["area_property_baseline"]
    assert model["mae"] < baseline["mae"]
    assert 0 <= report["test_interval_coverage"] <= 1
    assert report["model_reload_delta"] == 0.0
    print("Retained house-price evidence passed.")


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--self-test", action="store_true")
    parser.add_argument("--check-evidence", action="store_true")
    args = parser.parse_args()
    if args.self_test:
        self_test()
    if args.check_evidence:
        check_evidence()
    if not args.self_test and not args.check_evidence:
        parser.error("choose --self-test or --check-evidence")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## Canonical source: `src/__init__.py`


In [ ]:
"""UK house price prediction portfolio package."""


## Canonical source: `src/data.py`


In [ ]:
from __future__ import annotations

from pathlib import Path
from urllib.request import urlretrieve

import duckdb
import pandas as pd

SOURCE_URLS = {
    2025: "https://price-paid-data.publicdata.landregistry.gov.uk/pp-2025.csv",
    2026: "https://price-paid-data.publicdata.landregistry.gov.uk/pp-2026.csv",
}

MODEL_START = pd.Timestamp("2025-01-01")
VALIDATION_START = pd.Timestamp("2025-10-01")
TEST_START = pd.Timestamp("2026-01-01")
TEST_END = pd.Timestamp("2026-07-01")

CATEGORICAL_FEATURES = [
    "postcode_district",
    "postcode_area",
    "property_type",
    "old_new",
    "duration",
    "town_city",
    "district",
    "county",
]
NUMERIC_FEATURES = ["transfer_year", "transfer_month", "month_sin", "month_cos"]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES


def download_price_paid(cache_dir: Path) -> dict[int, Path]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    paths: dict[int, Path] = {}
    for year, url in SOURCE_URLS.items():
        path = cache_dir / f"pp-{year}.csv"
        if not path.exists():
            urlretrieve(url, path)
        paths[year] = path
    return paths


def load_residential(paths: dict[int, Path]) -> pd.DataFrame:
    """Load the exact modelling population used in the verified notebook."""
    con = duckdb.connect(database=":memory:")
    files = ",".join(repr(str(path)) for path in paths.values())
    con.execute(
        f"""
        CREATE VIEW price_paid AS
        SELECT
            column00 AS transaction_id,
            CAST(column01 AS BIGINT) AS price,
            CAST(strptime(column02, '%Y-%m-%d %H:%M') AS DATE) AS transfer_date,
            NULLIF(trim(column03), '') AS postcode,
            column04 AS property_type,
            column05 AS old_new,
            column06 AS duration,
            column11 AS town_city,
            column12 AS district,
            column13 AS county,
            column14 AS ppd_category,
            column15 AS record_status
        FROM read_csv([{files}], header=false, all_varchar=true)
        """
    )
    frame = con.execute(
        """
        SELECT
            transaction_id,
            price,
            transfer_date,
            postcode,
            split_part(postcode, ' ', 1) AS postcode_district,
            regexp_extract(postcode, '^[A-Z]+', 0) AS postcode_area,
            property_type,
            old_new,
            duration,
            town_city,
            district,
            county,
            EXTRACT(year FROM transfer_date)::INTEGER AS transfer_year,
            EXTRACT(month FROM transfer_date)::INTEGER AS transfer_month,
            sin(2*pi()*EXTRACT(month FROM transfer_date)/12) AS month_sin,
            cos(2*pi()*EXTRACT(month FROM transfer_date)/12) AS month_cos
        FROM price_paid
        WHERE property_type IN ('D', 'S', 'T', 'F')
          AND ppd_category = 'A'
          AND price BETWEEN 20000 AND 5000000
          AND postcode IS NOT NULL
          AND transfer_date >= DATE '2025-01-01'
          AND transfer_date < DATE '2026-07-01'
        ORDER BY transfer_date, transaction_id
        """
    ).df()
    con.close()
    frame["transfer_date"] = pd.to_datetime(frame["transfer_date"])
    for column in CATEGORICAL_FEATURES:
        frame[column] = frame[column].fillna("UNKNOWN").astype(str)
    return frame


def temporal_split(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train = frame[(frame.transfer_date >= MODEL_START) & (frame.transfer_date < VALIDATION_START)].copy()
    validation = frame[(frame.transfer_date >= VALIDATION_START) & (frame.transfer_date < TEST_START)].copy()
    test = frame[(frame.transfer_date >= TEST_START) & (frame.transfer_date < TEST_END)].copy()
    if train.empty or validation.empty or test.empty:
        raise ValueError("temporal split produced an empty partition")
    if train.transfer_date.max() >= validation.transfer_date.min():
        raise AssertionError("train/validation chronology is invalid")
    if validation.transfer_date.max() >= test.transfer_date.min():
        raise AssertionError("validation/test chronology is invalid")
    return train, validation, test


## Canonical source: `src/evaluation.py`


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def regression_metrics(actual, predicted) -> dict[str, float]:
    y = np.asarray(actual, dtype=float)
    p = np.asarray(predicted, dtype=float)
    return {
        "mae": float(mean_absolute_error(y, p)),
        "rmse": float(mean_squared_error(y, p) ** 0.5),
        "r2": float(r2_score(y, p)),
        "mape_pct": float(np.mean(np.abs((y - p) / y)) * 100),
        "within_20pct": float(np.mean(np.abs(y - p) / y <= 0.20)),
    }


class AreaPropertyBaseline:
    """Median baseline fitted on training data only.

    It uses postcode district + property type where available, falls back to
    property type, then to the global training median.
    """

    def fit(self, train: pd.DataFrame) -> "AreaPropertyBaseline":
        self.global_median = float(train.price.median())
        self.type_median = train.groupby("property_type").price.median()
        self.area_type_median = train.groupby(["postcode_district", "property_type"]).price.median()
        return self

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        keys = pd.MultiIndex.from_frame(frame[["postcode_district", "property_type"]])
        area = np.asarray(self.area_type_median.reindex(keys), dtype=float)
        fallback = frame.property_type.map(self.type_median).fillna(self.global_median).to_numpy(float)
        return np.where(np.isfinite(area), area, fallback)


def conformal_radius(actual, predicted, coverage: float = 0.90) -> float:
    if not 0 < coverage < 1:
        raise ValueError("coverage must be between zero and one")
    residual = np.abs(np.asarray(actual, float) - np.asarray(predicted, float))
    return float(np.quantile(residual, coverage, method="higher"))


def interval_coverage(actual, predicted, radius: float, floor: float = 20_000.0, cap: float = 5_000_000.0) -> dict[str, float]:
    y = np.asarray(actual, float)
    p = np.asarray(predicted, float)
    lower = np.clip(p - radius, floor, cap)
    upper = np.clip(p + radius, floor, cap)
    return {
        "coverage": float(np.mean((y >= lower) & (y <= upper))),
        "average_width_pounds": float(np.mean(upper - lower)),
    }


## Canonical source: `src/model.py`


In [ ]:
from __future__ import annotations

import numpy as np

from .data import CATEGORICAL_FEATURES, FEATURES

MODEL_CONFIG = {
    "iterations": 500,
    "depth": 9,
    "learning_rate": 0.08,
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "l2_leaf_reg": 8,
    "random_seed": 42,
    "allow_writing_files": False,
    "thread_count": -1,
    "verbose": False,
}


def build_model():
    from catboost import CatBoostRegressor

    return CatBoostRegressor(**MODEL_CONFIG)


def fit_model(train, validation):
    model = build_model()
    model.fit(
        train[FEATURES],
        np.log1p(train.price),
        cat_features=CATEGORICAL_FEATURES,
        eval_set=(validation[FEATURES], np.log1p(validation.price)),
        early_stopping_rounds=50,
    )
    return model


def predict_price(model, frame):
    return np.expm1(model.predict(frame[FEATURES])).clip(20_000, 5_000_000)


## Canonical source: `tests/test_evaluation.py`


In [ ]:
import numpy as np
import pandas as pd

from src.evaluation import AreaPropertyBaseline, conformal_radius, regression_metrics


def test_area_property_baseline_fallbacks():
    train = pd.DataFrame({
        "postcode_district": ["E1", "E1", "SW1"],
        "property_type": ["F", "F", "T"],
        "price": [300000, 320000, 700000],
    })
    test = pd.DataFrame({
        "postcode_district": ["E1", "N1", "N1"],
        "property_type": ["F", "F", "D"],
    })
    prediction = AreaPropertyBaseline().fit(train).predict(test)
    assert np.allclose(prediction, [310000, 310000, 320000])


def test_regression_metrics_are_exact_for_perfect_prediction():
    metrics = regression_metrics([100000, 200000], [100000, 200000])
    assert metrics["mae"] == 0.0
    assert metrics["rmse"] == 0.0
    assert metrics["r2"] == 1.0
    assert metrics["mape_pct"] == 0.0
    assert metrics["within_20pct"] == 1.0


def test_conformal_radius_uses_validation_errors_only():
    radius = conformal_radius([100, 200, 300, 400], [100, 180, 260, 450], coverage=0.75)
    assert radius == 50.0


# Portfolio depth check

**Meaningful code lines visible in this notebook:** 561. For a major recruiter-facing application the working target is roughly **1,000 meaningful lines**, with a practical guide of about 600–1,400 depending on the problem. This notebook is below the major-project guide and should grow only through substantive analysis/application depth.

Line count is not a quality metric by itself. Add code only when it improves the real project: data acquisition, validation, cleaning, EDA, visualisation, feature engineering, baselines, model comparison, tuning, leakage control, error analysis, explainability, uncertainty, inference, tests, monitoring, deployment or decision logic.
